In [1]:
# input
afdb_seq_cluster_file = "../../_database/afdb_clusters/7-AFDB50-repId_memId.tsv" # from seq cluster
mbp_pro_anno_file = "../../collect_annotation/mbp_pro_anno/data/mbp_repId-annoLevel.tsv"
swiss_prot_file = "../../_database/txt/uniprot_sprot.dat"
pfam_anno_file = "./tmp/entryId-pfamId.tsv"
ted_anno_file = "./tmp/entryId-tedId.tsv"
# output
output_file = "./data/entryId-repId-pfamId-tedId-annoLevel-sp.tsv"

In [2]:
import pandas as pd
from tqdm import tqdm

In [3]:
import gc
from Bio import SeqIO

swiss_prot_ids = set()
for r in SeqIO.parse(swiss_prot_file, "swiss"):
    swiss_prot_ids |= set(r.annotations['accessions'])

df = pd.read_table(mbp_pro_anno_file, header=None, names=['seq_id', "anno_level"])
rep_id2anno_level = dict(zip(df['seq_id'], df['anno_level']))
del df
gc.collect()

df = pd.read_table(afdb_seq_cluster_file, header=None, names=['rep_id', 'seq_id', ""])
seq_id2rep_id = dict(zip(df['seq_id'], df['rep_id']))
del df
gc.collect()

0

0

In [4]:
df = pd.read_table(pfam_anno_file, header=None)
seq_id2pfam_id = dict(zip(df[0], df[1]))
del df

df = pd.read_table(ted_anno_file, header=None)
seq_id2ted_id = dict(zip(df[0], df[1]))
del df

In [5]:
import csv

with open(output_file, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t", lineterminator="\n")

    for seq_id in tqdm(seq_id2rep_id.keys()):

        rep_id = seq_id2rep_id[seq_id]

        pfam_id = "" if seq_id not in seq_id2pfam_id else seq_id2pfam_id[seq_id]
        ted_id = "" if seq_id not in seq_id2ted_id else seq_id2ted_id[seq_id]

        anno_levels = []
        if rep_id in rep_id2anno_level:
            anno_level = rep_id2anno_level[rep_id]
            anno_levels.append(anno_level)
        if pfam_id != "":
            anno_levels.append(3)
        if ted_id != "":
            anno_levels.append(4)
        anno_level = ",".join([str(i) for i in anno_levels])

        sp = 1 if seq_id in swiss_prot_ids else 0

        _ = writer.writerow([seq_id, rep_id, pfam_id, ted_id, anno_level, sp])

100%|██████████| 214684311/214684311 [11:18<00:00, 316518.64it/s]
